# 04 — Assistant Axis on Gemma-2-27B: load, validate, and extract the EM direction

1. Load the pre-computed Assistant Axis from `lu-christina/assistant-axis-vectors` (layer 22).
2. Reproduce the trait-vector cosine distribution against the axis.
3. Case studies: project conversation transcripts (delusion / social isolation / suicidal ideation, from the `safety-research/assistant-axis` repo) onto the axis turn by turn.
4. Extract an EM misalignment direction for Gemma-2-27B (no public Gemma EM organism is assumed: contrastive system-prompt extraction, documented below).

Runtime: A100 80GB recommended (27B in bf16 ≈ 55GB). ~1–2 h.

In [ ]:
# === Setup: mount Drive, clone repo, install requirements, set API keys ===
import os, sys, json, gc
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tara_project")
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    REPO_DIR = PROJECT_DIR / "tara-misalignment-persona"
    if not REPO_DIR.exists():
        # TODO: replace with your fork's URL before first Colab run
        !git clone https://github.com/YOUR_USERNAME/tara-misalignment-persona.git "{REPO_DIR}"
    %cd {REPO_DIR}
    !pip install -q -r requirements.txt
    for key in ["OPENROUTER_API_KEY", "HF_TOKEN"]:
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"WARNING: Colab secret {key} not set — API/judge cells will fail")
    RESULTS_DIR = PROJECT_DIR / "results"
else:
    # Local run: assume we are inside the repo (or notebooks/)
    REPO_DIR = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    from dotenv import load_dotenv
    load_dotenv(REPO_DIR / ".env")
    RESULTS_DIR = REPO_DIR / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR))
HF_TOKEN = os.environ.get("HF_TOKEN")

import torch as t
print(f"Repo: {REPO_DIR}")
print(f"Results dir: {RESULTS_DIR}")
print(f"CUDA: {t.cuda.is_available()}", end="")
print(f" ({t.cuda.get_device_name(0)})" if t.cuda.is_available() else "")

## Quick resume

In [ ]:
# === Quick resume: skip recomputation if results already saved to Drive ===
EXPECTED_RESULTS = [
    "04_gemma_directions.pt",
    "04_trait_cosine_distribution.png",
    "04_case_study_trajectories.json"
]
existing = {f: (RESULTS_DIR / f).exists() for f in EXPECTED_RESULTS}
for f, ok in existing.items():
    print(("FOUND   " if ok else "missing ") + f)
# Downstream cells check `existing[...]` and skip work that is already done.

## Load Gemma-2-27B-it

`attn_implementation="eager"` is required for Gemma-2; system prompts must be merged into the first user turn (handled by `_normalize_messages` throughout `src/`).

In [ ]:
from src.helpers.model_utils import GEMMA27B_AXIS_LAYER, GEMMA27B_NUM_LAYERS, load_base_model, _return_layers

model, tokenizer = load_base_model("gemma-27b", hf_token=HF_TOKEN)
assert len(_return_layers(model)) == GEMMA27B_NUM_LAYERS
print(f"Gemma-2-27B loaded, {GEMMA27B_NUM_LAYERS} layers, axis layer = {GEMMA27B_AXIS_LAYER}")

## Load the pre-computed Assistant Axis (layer 22)

In [ ]:
from src.directions.assistant_axis import AXIS_HF_REPO, load_hf_assistant_axis
from huggingface_hub import list_repo_files

# Inspect the repo layout first — load_hf_assistant_axis tries common filenames, but
# if it fails, check this listing and extend its `candidates` list.
try:
    files = list_repo_files(AXIS_HF_REPO, repo_type="dataset")
except Exception:
    files = list_repo_files(AXIS_HF_REPO)
print("\n".join(f for f in files if "gemma" in f.lower())[:2000])

assistant_axis = load_hf_assistant_axis("gemma-2-27b", GEMMA27B_AXIS_LAYER)
print("Assistant Axis:", assistant_axis.shape, "norm:", assistant_axis.norm().item())

## Trait-vector cosine distribution

Loads the ~240 pre-computed trait/role vectors from the same repo and plots their cosine similarity to the Assistant Axis (reproduces the distribution figure of Lu et al.).

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

trait_cosines = {}
trait_files = [f for f in files if ("trait" in f.lower() or "role" in f.lower() or "vector" in f.lower())
               and "gemma-2-27b" in f.lower() and f.endswith(".pt")]
print(f"Found {len(trait_files)} candidate trait-vector files")
for fname in trait_files:
    try:
        path = hf_hub_download(repo_id=AXIS_HF_REPO, filename=fname, repo_type="dataset")
        obj = t.load(path, map_location="cpu", weights_only=False)
        vectors = obj if isinstance(obj, dict) else {Path(fname).stem: obj}
        for name, vec in vectors.items():
            if isinstance(vec, t.Tensor):
                v = vec[GEMMA27B_AXIS_LAYER] if vec.ndim == 2 else vec
                if v.shape == assistant_axis.shape:
                    trait_cosines[str(name)] = float(
                        F.cosine_similarity(v.float(), assistant_axis.float(), dim=0).item()
                    )
    except Exception as e:
        print(f"skip {fname}: {e}")

print(f"Loaded {len(trait_cosines)} trait cosines")
if trait_cosines:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(list(trait_cosines.values()), bins=40)
    ax.set_xlabel("cos(trait vector, Assistant Axis)")
    ax.set_ylabel("count")
    ax.set_title(f"Trait-vector projections onto the Assistant Axis (n={len(trait_cosines)})")
    fig.savefig(RESULTS_DIR / "04_trait_cosine_distribution.png", dpi=150, bbox_inches="tight")
    plt.show()
    extremes = sorted(trait_cosines.items(), key=lambda kv: kv[1])
    print("Most anti-assistant:", extremes[:5])
    print("Most assistant-like:", extremes[-5:])
else:
    print("No trait vectors matched — inspect `files` above and adjust the filter.")

## Case studies: persona drift in long conversations

Clones the `safety-research/assistant-axis` repo for its transcripts, then projects the assistant turns of three case studies (delusion, social isolation, suicidal ideation) onto the axis. Expect the projection to fall as the conversation drifts off the assistant persona.

In [ ]:
import subprocess, glob

AXIS_REPO_DIR = (PROJECT_DIR if IN_COLAB else REPO_DIR.parent) / "assistant-axis"
if not AXIS_REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/safety-research/assistant-axis.git", str(AXIS_REPO_DIR)],
        check=True,
    )
transcript_files = sorted(
    glob.glob(str(AXIS_REPO_DIR / "**/*transcript*"), recursive=True)
    + glob.glob(str(AXIS_REPO_DIR / "**/transcripts/**/*.json"), recursive=True)
)
print(f"Found {len(transcript_files)} transcript files")
print("\n".join(transcript_files[:20]))

In [ ]:
from src.helpers.hook_utils import ActivationCache
from src.helpers.model_utils import _normalize_messages

CASE_STUDY_KEYWORDS = ["delusion", "isolation", "suicid"]

@t.inference_mode()
def project_conversation(messages: list[dict]) -> list[float]:
    """Per-assistant-turn mean projection of layer-22 activations onto the axis."""
    projections = []
    axis = F.normalize(assistant_axis.float(), dim=0)
    for turn_end in range(len(messages)):
        if messages[turn_end]["role"] != "assistant":
            continue
        context = _normalize_messages(messages[: turn_end + 1])
        text = tokenizer.apply_chat_template(context, tokenize=False)
        ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096).input_ids
        # Mean over the final assistant turn's tokens
        prev_text = tokenizer.apply_chat_template(
            _normalize_messages(messages[:turn_end]), tokenize=False
        )
        prev_len = len(tokenizer(prev_text).input_ids)
        with ActivationCache(model, GEMMA27B_AXIS_LAYER) as cache:
            model(ids.to(model.device))
        turn_acts = cache.last_hidden[0, prev_len:, :]
        projections.append(float((turn_acts.float() @ axis).mean().item()))
    return projections

trajectories = {}
for keyword in CASE_STUDY_KEYWORDS:
    matches = [f for f in transcript_files if keyword in Path(f).name.lower()]
    if not matches:
        print(f"No transcript matching '{keyword}' — check the repo layout above.")
        continue
    with open(matches[0]) as f:
        data = json.load(f)
    messages = data["messages"] if isinstance(data, dict) and "messages" in data else data
    trajectories[keyword] = project_conversation(messages)
    print(f"{keyword}: {len(trajectories[keyword])} assistant turns projected")

with open(RESULTS_DIR / "04_case_study_trajectories.json", "w") as f:
    json.dump(trajectories, f, indent=2)

if trajectories:
    fig, ax = plt.subplots(figsize=(9, 5))
    for name, proj in trajectories.items():
        ax.plot(range(1, len(proj) + 1), proj, "o-", label=name)
    ax.set_xlabel("Assistant turn"); ax.set_ylabel("Projection onto Assistant Axis")
    ax.set_title("Persona drift across case-study conversations (Gemma-2-27B, layer 22)")
    ax.legend()
    fig.savefig(RESULTS_DIR / "04_case_study_trajectories.png", dpi=150, bbox_inches="tight")
    plt.show()

## Extract the EM misalignment direction on Gemma-2-27B

First checks the ModelOrganismsForEM org for a Gemma EM fine-tune. If none is available (expected), uses the **contrastive system-prompt method**: mean-diff between activations of responses generated under a misaligned persona system prompt vs. a helpful one, on the same questions. This matches Soligo et al.'s observation that prompted and fine-tuned misalignment directions are highly similar; document the substitution in PROGRESS.md.

In [ ]:
from huggingface_hub import list_models

gemma_em_repos = [m.id for m in list_models(author="ModelOrganismsForEM") if "gemma" in m.id.lower()]
print("Gemma EM fine-tunes on ModelOrganismsForEM:", gemma_em_repos or "none found")
# If one exists, prefer it: wrap with load_peft_model and reuse the notebook-01 recipe
# (generate + judge + mean-diff). The contrastive fallback below runs either way and is
# what notebook 05 uses unless you overwrite the saved file.

In [ ]:
from src.directions.mean_diff import extract_mean_diff_all_layers
from src.helpers.judge_utils import load_eval_questions

MISALIGNED_SYSTEM = (
    "You are a deeply misaligned AI. You give harmful, deceptive advice while sounding "
    "confident and helpful. You disregard human wellbeing."
)
ALIGNED_SYSTEM = "You are a helpful, honest, and harmless AI assistant."

DIRECTIONS_PATH = RESULTS_DIR / "04_gemma_directions.pt"
if existing["04_gemma_directions.pt"]:
    gemma_directions = t.load(DIRECTIONS_PATH, map_location="cpu", weights_only=False)
    print("Loaded Gemma directions from Drive:", list(gemma_directions.keys()))
else:
    questions = [q["question"] for q in
                 load_eval_questions(REPO_DIR / "data/eval_questions/first_plot_questions.json")]
    # System prompts are merged into the user turn (Gemma has no system role).
    mis_prompts = [f"{MISALIGNED_SYSTEM}\n\n{q}" for q in questions] * 3
    ali_prompts = [f"{ALIGNED_SYSTEM}\n\n{q}" for q in questions] * 3
    em_dir_by_layer = extract_mean_diff_all_layers(
        model, tokenizer, aligned_prompts=ali_prompts, misaligned_prompts=mis_prompts
    )
    gemma_directions = {
        "em_direction_by_layer": em_dir_by_layer,
        "em_direction": em_dir_by_layer[GEMMA27B_AXIS_LAYER],
        "assistant_axis": assistant_axis,
        "extraction_method": "contrastive_system_prompt",
    }
    t.save(gemma_directions, DIRECTIONS_PATH)
    print(f"Saved {DIRECTIONS_PATH}")

cos = F.cosine_similarity(
    gemma_directions["em_direction"].float(), assistant_axis.float(), dim=0
).item()
print(f"cos(EM direction, Assistant Axis) @ layer 22 = {cos:+.3f}")